# Stacking Energy and Band Gap of Graphene on h-BN

> **Gianluca Giovannetti, Petr A. Khomyakov, Geert Brocks, Paul J. Kelly & Jeroen van den Brink**
> Substrate-induced band gap in graphene on hexagonal boron nitride: Ab initio density functional calculations. Physical Review B, 76, 073103. 2007.
> [https://doi.org/10.1103/PhysRevB.76.073103](https://doi.org/10.1103/PhysRevB.76.073103)

Calculate the total energy and the band structure of the seven graphene/h-BN stackings at d = 3.4 Å
created in the [structure notebook](interface_2d_2d_boron_nitride_graphene.ipynb), using Quantum
ESPRESSO and the band structure + density of states workflow from Standata, and compare the
ordering of the sliding energies and the direct gap at K with Giovannetti et al. (2007).

The seven stackings follow the sliding path of Fig. 7(a) in Jung et al. (2015). That figure is a model
curve parameterised from RPA calculations; the DFT numbers compared here are Giovannetti's.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_2d_2d_boron_nitride_graphene.ipynb), which saves them to the `uploads` folder under the names used in cell 1.2 below.
1. Set the materials and the calculation parameters in cells 1.2 and 1.3, or keep the default values: everything runs with them.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results; section 8 prints the comparison table and plots.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the materials, workflow, model, compute resources and jobs.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder, print their provenance and save them to the platform.
1. Configure the workflow: select the application and the model, load the band structure + DOS workflow from Standata, and set the computational parameters for each material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run one job per material, one after another, re-using a job that already ran under the same workflow name.
1. Retrieve results: band structures, total energies and the direct gap along the path.
1. Compare with Giovannetti et al. (2007).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

# Names saved by the structure notebook; the symmetric stackings carry their Jung 2015 / Giovannetti 2007 label
MATERIALS = {
    "Gr/hBN d3.4 shift 0of6 BA": {"shift": 0, "stacking": "BA"},
    "Gr/hBN d3.4 shift 1of6": {"shift": 1, "stacking": None},
    "Gr/hBN d3.4 shift 2of6 AA": {"shift": 2, "stacking": "AA"},
    "Gr/hBN d3.4 shift 3of6": {"shift": 3, "stacking": None},
    "Gr/hBN d3.4 shift 4of6 AB": {"shift": 4, "stacking": "AB"},
    "Gr/hBN d3.4 shift 5of6": {"shift": 5, "stacking": None},
    "Gr/hBN d3.4 shift 6of6 BA": {"shift": 6, "stacking": "BA"},
}

WORKFLOW_SEARCH_TERM = "band_structure_dos.json"
MY_WORKFLOW_NAME = "Band Structure + DOS"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1
TIME_LIMIT = "01:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set the DFT parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # Giovannetti et al. 2007 use LDA: GGA gives essentially no interlayer binding
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the only LDA family the platform publishes for B, C and N
ECUTWFC = 40   # Ry, GBRV's tested cutoff
ECUTRHO = 200  # Ry, GBRV's tested charge-density cutoff

KGRID = [36, 36, 1]  # Giovannetti et al. 2007; a multiple of 3 keeps K on the mesh
SMEARING_SETTINGS = {"degauss": 0.001}  # Ry; the gaps compared are 30-80 meV
KPATH_STEPS = 40
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KGRID[0]} p{KPATH_STEPS} g{SMEARING_SETTINGS['degauss']}"

SCF_UNIT = "pw_scf"
NSCF_UNIT = "pw_nscf"
BANDS_UNIT = "pw_bands"
NUMBER_OF_OCCUPIED_BANDS = 8  # 16 valence electrons: C 4 + 4, B 3, N 5

KPATH = [
    {"point": "Γ", "steps": KPATH_STEPS},
    {"point": "K", "steps": KPATH_STEPS},
    {"point": "M", "steps": KPATH_STEPS},
    {"point": "Γ", "steps": 1},
]

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

The structures, their geometry and their lattice type all come from the structure notebook; this one
only loads them by name.

In [ ]:
from collections import Counter
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.notebooks_utils.core.entity.material.api import load_material

materials = {}
for name, settings in MATERIALS.items():
    material = load_material(client, FOLDER, name, ACCOUNT_ID)
    materials[name] = material
    composition = "".join(
        f"{element}{count}" for element, count in sorted(Counter(material.basis.elements.values).items()))
    interlayer_distance = get_average_interlayer_distance(
        material, InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value)
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, a = {material.lattice.a:.4f} Å, "
          f"gamma = {material.lattice.gamma:.3f}°, interlayer distance = {interlayer_distance:.3f} Å, "
          f"{settings['stacking'] or 'bridge'}")

### 3.2. Preview the materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": name} for name, material in materials.items()],
                    viewer="wave")

### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = Material.create(get_or_create_material(client, material, ACCOUNT_ID))

## 4. Configure the workflow
### 4.1. Select application and model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. Configure the workflow

One workflow per material, all with the settings of 1.3. Each workflow is named after its material
and `MODEL_TAG`; section 6 finds a job that already ran by that name.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

from copy import deepcopy

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
grid_context = PointsGridDataProvider(
    material=next(iter(materials.values())), dimensions=KGRID, isEdited=True).get_context_item_data()
path_context = PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data()

workflows = {}
for name in MATERIALS:
    workflow = Workflow.create(deepcopy(workflow_config))
    workflow.name = f"{MY_WORKFLOW_NAME} {name} {MODEL_TAG}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model

    for unit_name, contexts in [(SCF_UNIT, [grid_context, cutoffs_context]),
                                (NSCF_UNIT, [grid_context, cutoffs_context]),
                                (BANDS_UNIT, [path_context, cutoffs_context])]:
        unit = subworkflow.get_unit_by_name(name=unit_name)
        for context in contexts:
            unit.add_context(context)
        subworkflow.set_unit(unit)
    patch_workflow_qe_input(workflow, {"system": SMEARING_SETTINGS}, [SCF_UNIT, NSCF_UNIT, BANDS_UNIT])
    workflows[name] = workflow
    print(workflow.name)

### 4.3. Preview the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

visualize_workflow(workflows[next(iter(MATERIALS))])

### 4.4. Save the workflows to the collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

for name, workflow in workflows.items():
    saved = Workflow.create(get_or_create_workflow(client, workflow, ACCOUNT_ID))
    print(f"{name}: workflow {saved.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Run the jobs one at a time

A job is found by its material and its workflow name, which carries the model tag, and re-used if it
exists; otherwise it is created and submitted. The jobs run one after another.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job_ids = {}
for name, workflow in workflows.items():
    job = find_job_for_material(
        client, saved_materials[name].id, workflow.name, ACCOUNT_ID,
        statuses=("submitted", "queued", "active", "finished"),
    )
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing job {job['_id']}")
    await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
    job_ids[name] = job["_id"]

## 7. Retrieve the results
### 7.1. Band structures

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structures = {}
for name, job_id in job_ids.items():
    band_structures[name] = get_properties_for_job(client, job_id, property_name="band_structure")
    visualize_properties(band_structures[name], title=f"Band Structure: {name}",
                         extra_config={"material": materials[name].to_dict()})

### 7.2. Total energies

In [ ]:
total_energies = {}
for name, job_id in job_ids.items():
    total_energies[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    print(f"{name}: {total_energies[name]:.6f} eV")

### 7.3. Direct gap along the path

The smallest difference along Γ–K–M–Γ between the lowest unoccupied and the highest occupied band;
the k-point where it occurs is printed in crystal coordinates. For the three symmetric stackings it
sits at K = (1/3, 1/3, 0).

In [ ]:
import numpy as np

gaps = {}
for name, band_structure in band_structures.items():
    bands = np.array(band_structure[0]["yDataSeries"])
    gaps_along_path = bands[NUMBER_OF_OCCUPIED_BANDS] - bands[NUMBER_OF_OCCUPIED_BANDS - 1]
    kpoint_index = int(np.argmin(gaps_along_path))
    gaps[name] = 1000 * gaps_along_path[kpoint_index]
    kpoint = band_structure[0]["xDataArray"][kpoint_index]
    print(f"{name}: {gaps[name]:.1f} meV at k = ({kpoint[0]:.4f}, {kpoint[1]:.4f}, {kpoint[2]:.4f})")

## 8. Compare with Giovannetti et al. (2007)

ΔE is the total energy relative to the first entry of `MATERIALS`. The paper's AA and BA gaps are read
off its Fig. 4 at d = 3.4 Å, and AB is the paper's value at its 3.40 Å equilibrium; the gaps at its
own equilibrium distances are printed for reference.

Settings that differ from the paper's: cell a = 2.509 Å (h-BN unstrained, graphene +1.79 %) against 2.445 Å; one h-BN layer against four; GBRV ultrasoft pseudopotentials against VASP at 600 eV; Gaussian smearing against the tetrahedron method; no dipole correction.

In [ ]:
from mat3ra.notebooks_utils.plot import plot_series

# meV; AA and BA read off Fig. 4 at d = 3.4 Å (±5 meV), AB is the paper's value at its 3.40 Å equilibrium
GIOVANNETTI_GAPS_AT_3_4_ANGSTROM = {"AA": 80, "AB": 46, "BA": 30}
GIOVANNETTI_GAPS_AT_EQUILIBRIUM = {"AA": (3.50, 56), "AB": (3.40, 46), "BA": (3.22, 53)}

reference_name = next(iter(MATERIALS))
series = [
    {
        "shift": settings["shift"],
        "stacking": settings["stacking"] or "bridge",
        "energy": 1000 * (total_energies[name] - total_energies[reference_name]),
        "gap": gaps[name],
    }
    for name, settings in MATERIALS.items()
]
print(f"{'shift':>5} {'stacking':>8} {'ΔE (meV)':>9} {'gap (meV)':>10} {'paper gap (meV)':>16} {'deviation':>10}")
for item in series:
    paper_gap = GIOVANNETTI_GAPS_AT_3_4_ANGSTROM.get(item["stacking"])
    deviation = f"{(item['gap'] - paper_gap) / paper_gap:+.0%}" if paper_gap else ""
    print(f"{item['shift']:>5} {item['stacking']:>8} {item['energy']:>9.2f} {item['gap']:>10.1f} "
          f"{paper_gap or '':>16} {deviation:>10}")

plot_series(series=series, x_key="shift", y_key="energy", xlabel="Shift from BA along y (steps of a/(2√3))",
            ylabel="ΔE (meV per cell)", title="Total energy along the sliding path")
plot_series(series=series, x_key="shift", y_key="gap", xlabel="Shift from BA along y (steps of a/(2√3))",
            ylabel="Direct gap along the path (meV)", title="Direct gap along the sliding path")

print("Gaps at the paper's equilibrium distances: " + ", ".join(
    f"{stacking} {distance:.2f} Å {gap} meV" for stacking, (distance, gap) in GIOVANNETTI_GAPS_AT_EQUILIBRIUM.items()))

## References

[1] G. Giovannetti, P. A. Khomyakov, G. Brocks, P. J. Kelly and J. van den Brink, "Substrate-induced band gap in graphene on hexagonal boron nitride: Ab initio density functional calculations", Phys. Rev. B 76, 073103 (2007). https://doi.org/10.1103/PhysRevB.76.073103

[2] J. Jung, A. M. DaSilva, A. H. MacDonald and S. Adam, "Origin of band gaps in graphene on hexagonal boron nitride", Nat. Commun. 6, 6308 (2015). https://doi.org/10.1038/ncomms7308